In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression,Ridge,Lasso

In [ ]:
data = pd.read_csv('/home/san/studentperformancepredictor/datasets/StudentPerformanceFactors.csv')

In [4]:
data.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [5]:
X = data.drop('Exam_Score',axis = 1)
y = data['Exam_Score']

In [11]:
# Identifying the Coulmn Types
numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
print(numeric_features)
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
print(categorical_features)

['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)

In [17]:
# Define Preprocessing steps of ColumnTransformers and OneHotEncoding
preprocessor = ColumnTransformer(transformers =[
    ('numeric',StandardScaler(),numeric_features),
    ('categorical',OneHotEncoder(),categorical_features)
])

# pipeline of thw workflow
pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

# fit the model
pipeline.fit(X_train,y_train)

#predict
y_pred = pipeline.predict(X_test)

#Evaluate
mse = mean_squared_error(y_test,y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Predicted grade: {y_pred[0]:,.0f}")

Mean Squared Error: 3.25
Predicted grade: 65


In [19]:
# cross Validating
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(pipeline, X, y, cv=3, scoring='neg_mean_squared_error')
print(f"Average MSE (CV): {np.mean(-scores):.2f}")

Average MSE (CV): 4.18


In [24]:
# Define models in a dictionary
models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0, max_iter=10000)
}

# Evaluate each with cross-validation
print("Cross-validated MSE scores:\n")
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),  
        ('regressor', model)
    ])
    
    neg_mse_scores = cross_val_score(pipeline, X, y, cv=3, scoring='neg_mean_squared_error')
    mse_scores = -neg_mse_scores
    print(f"{name} Regression - Mean MSE: {np.mean(mse_scores):.2f}, Std: {np.std(mse_scores):.2f}")


Cross-validated MSE scores:

Linear Regression - Mean MSE: 4.18, Std: 0.97
Ridge Regression - Mean MSE: 4.18, Std: 0.97
Lasso Regression - Mean MSE: 8.97, Std: 0.68


### Linear Regression and Ridge is giving Similar mean MSE, which states that the Models arent Overfitting, but Lasso is performing significantly worse, as it may be shrinking coefficients to zero,so useful Columns might have been dropped. For lasso aplha = 1.0 is too high. We can apply grid search to find out which the perfect values for alpha.

In [28]:
# Define range of alpha values to test
alpha_values = {'regressor__alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

# Ridge regression with GridSearchCV
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

ridge_grid = GridSearchCV(ridge_pipeline, alpha_values, cv=3, scoring='neg_mean_squared_error')
ridge_grid.fit(X, y)
print(f"Best Ridge alpha: {ridge_grid.best_params_['regressor__alpha']}")
print(f"Best Ridge MSE: {-ridge_grid.best_score_:.2f}")

# Lasso regression with GridSearchCV
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=10000))
])

lasso_grid = GridSearchCV(lasso_pipeline, alpha_values, cv=3, scoring='neg_mean_squared_error')
lasso_grid.fit(X, y)
print(f"Best Lasso alpha: {lasso_grid.best_params_['regressor__alpha']}")
print(f"Best Lasso MSE: {-lasso_grid.best_score_:.2f}")

Best Ridge alpha: 10
Best Ridge MSE: 4.18
Best Lasso alpha: 0.001
Best Lasso MSE: 4.18


### so it suggests the alpha for lasso be 0.001 and ridge is 10